## 과제 2: 실시간 추론용으로 모델 배포

기계 학습(ML) 모델을 빌드, 훈련 및 평가하여 의도한 비즈니스 문제를 해결할 수 있음을 확인한 후에 비즈니스 운영에서 의사 결정을 용이하게 하도록 해당 모델을 배포할 수 있습니다. Amazon SageMaker는 짧은 지연 시간 및 높은 처리량에서 장기 실행 추론 작업에 이르는 다양한 배포 옵션을 제공합니다. SageMaker 추론을 사용하여 추론을 반환하는 엔드포인트를 설정하거나 모델에서 배치 추론을 실행할 수 있습니다.

SageMaker는 워크로드에 가장 적합한 옵션을 선택할 수 있도록 여러 추론 옵션을 제공합니다. 이 실습에서는 실시간 추론을 사용합니다. 특히, 실시간 추론은 일관되게 낮은 지연 시간과 처리량에 민감한 워크로드를 유지할 때 모델을 호스팅하기 위한 유용한 옵션입니다. 선택한 인스턴스 유형을 통해 지원되며 지속적인 트래픽을 처리할 수 있는 일관된 완전관리형 엔드포인트(REST API)를 위해 실시간 추론을 사용합니다. 실시간 추론은 최대 6MB의 페이로드 크기와 60초의 처리 시간을 지원할 수 있습니다.

배포를 위해 모델을 준비하려면 프로덕션(변형)에서 하나 이상의 모델의 이름과 각 프로덕션 변형을 호스팅하기 위해 SageMaker에서 시작할 ML 컴퓨팅 인스턴스를 지정하는 엔드포인트 구성을 정의합니다. 프로덕션에서 모델을 호스팅할 때 엔드포인트를 구성하여 배포된 기계 학습 컴퓨팅 인스턴스를 탄력적으로 크기를 조정할 수 있습니다. 각 프로덕션 변형에 대해 배포하려고 하는 기계 학습 컴퓨팅 인스턴스의 수를 지정합니다. 2개 이상의 인스턴스를 지정할 경우 SageMaker는 여러 가용 영역에서 인스턴스를 시작합니다. 이렇게 하면 지속적인 가용성이 보장됩니다. 인스턴스 배포는 SageMaker가 관리합니다. 모델 및 엔드포인트 구성이 있는 경우 [CreateEndpoint API](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateEndpoint.html)를 사용하여 엔드포인트를 생성합니다. 엔드포인트 구성을 SageMaker에 제공합니다. 그러면 이 서비스가 기계 학습 컴퓨팅 인스턴스를 시작하고 구성에 지정된 대로 하나 이상의 모델을 배포합니다. 

이 과제에서는 모델을 배포하고, 엔드포인트를 생성하고, 엔드포인트에서 실시간 추론을 실행합니다.

### 과제 2.1: 환경 설정

모델 튜닝을 시작하기 전에 필요한 종속 항목을 모두 설치하십시오.

In [ ]:
%pip install -q "sagemaker>=2,<3"
# Install dependencies
%pip install seaborn
%reset -f
import boto3
import pandas as pd
import sagemaker
import time
import numpy as np
import seaborn as sns

from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

role = sagemaker.get_execution_role()
region = boto3.Session().region_name
sess = boto3.Session()
sm = sess.client('sagemaker')
sm_runtime = sess.client("sagemaker-runtime")
s3_client = boto3.client("s3")

그런 다음, 데이터 집합을 가져옵니다.

In [ ]:
# Import the dataset 
s3 = boto3.resource('s3')
for buckets in s3.buckets.all():
    if 'labdatabucket' in buckets.name:
        bucket = buckets.name
print("Bucket: ", bucket)
prefix = 'scripts/data'

# Download the file to use for inference later in the notebook
s3_client.download_file(bucket, f'{prefix}/test/adult_data_processed_test.csv', 'adult_data_processed_test.csv')

# Open adult_data_processed_test.csv with pandas, save the first column as df_labels, remove the labels from the dataset, and save it back to adult_data_processed_test.csv
df = pd.read_csv('adult_data_processed_test.csv')
df_labels = df.iloc[:, 0]
df = df.drop(df.columns[0], axis=1)
df.to_csv('adult_data_processed_test_no_target.csv', index=False, header=False)

그런 후 이전 실습에서 선택한 최선의 모델을 가져옵니다. 실습 환경 생성 중에 **model.tar.gz** 파일이 업로드되었습니다. 

In [ ]:
# Upload the model to your Amazon S3 bucket
s3_client.upload_file(
    Filename="Lab4Repository/model.tar.gz", Bucket=bucket, Key=f"{prefix}/models/model.tar.gz"
)

# Set a date to use in the model name
create_date = time.strftime("%Y-%m-%d-%H-%M-%S")
model_name = 'income-model-{}'.format(create_date)

# Retrieve the container image
container = sagemaker.image_uris.retrieve(
    region=region, 
    framework='xgboost', 
    version='1.5-1'
)

# Set up the model
income_model = sm.create_model(
    ModelName = model_name,
    ExecutionRoleArn = role,
    PrimaryContainer = {
        'Image': container,
        'ModelDataUrl': f's3://{bucket}/{prefix}/models/model.tar.gz',
    }
)

## 과제 2.2: 엔드포인트 구성

이전 실습에서 모델은 이미 생성했을 것입니다. 이제 엔드포인트 구성을 생성할 수 있습니다. 

사용하려는 엔드포인트 구성 이름과 인스턴스 유형을 설정합니다.

엔드포인트 구성을 생성하려면 다음 옵션을 설정해야 합니다.
- **VariantName**: 프로덕션 변형(프로덕션 환경의 모델 하나 이상)의 이름입니다.
- **ModelName**: 호스팅할 모델의 이름입니다. 모델을 생성할 때 지정한 이름을 사용하면 됩니다.
- **InstanceType**: 컴퓨팅 인스턴스 유형입니다.
- **InitialInstanceCount**: 처음 시작할 인스턴스의 수입니다.

엔드포인트의 입력과 SageMaker 실시간 엔드포인트의 추론 출력을 Amazon S3에 로깅하려는 경우 데이터 캡처 기능을 사용하도록 설정할 수 있습니다. 데이터 캡처는 일반적으로 훈련, 디버깅 및 모니터링에 사용할 수 있는 정보를 기록하는 데 사용됩니다. 데이터 캡처가 사용하도록 설정되어 있으면 Amazon SageMaker Studio에서 엔드포인트를 탐색할 때 엔드포인트 관련 추가 세부 정보가 표시됩니다. 이 실습 뒷부분에서 데이터 캡처 기능을 구성하여 데이터 캡처를 사용하도록 설정하는 방법을 확인합니다.

데이터 캡처 기능을 추가하는 방법에 관한 자세한 내용은 [데이터 캡처](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html)를 참조하십시오.

In [ ]:
# Create an endpoint config name. Here you create one based on the date  
# so it you can search endpoints based on creation time.
endpoint_config_name = 'income-model-real-time-endpoint-{}'.format(create_date)                              
instance_type = 'ml.m5.4xlarge'   
initial_sampling_percentage = 25 # Choose a value between 0 and 100
capture_modes = [ "Input",  "Output" ] # Specify input, output, or both

endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name, # You will specify this name in a CreateEndpoint request.
    # List of ProductionVariant objects, one for each model that you want to host at this endpoint.
    ProductionVariants=[
        {
            "VariantName": "variant1", # The name of the production variant.
            "ModelName": model_name, 
            "InstanceType": instance_type, # Specify the compute instance type.
            "InitialInstanceCount": 1 # Number of instances to launch initially.
        }
    ],
    DataCaptureConfig= {
        'EnableCapture': True, # Whether data should be captured or not.
        'InitialSamplingPercentage' : initial_sampling_percentage,
        'DestinationS3Uri': f's3://{bucket}/data-capture',
        'CaptureOptions': [{"CaptureMode" : capture_mode} for capture_mode in capture_modes]
    }
)

print(f"Created EndpointConfig: {endpoint_config_response['EndpointConfigArn']}")

## 과제 2.3: 엔드포인트 생성

다음으로는 엔드포인트를 생성합니다. 실시간 엔드포인트를 생성하면 SageMaker가 ML 컴퓨팅 인스턴스를 시작하고 구성에 지정된 대로 모델을 하나 이상 배포합니다. 이 실습에서는 추론용 모델 하나만 배포합니다. SageMaker에서는 다중 모델 엔드포인트를 생성할 수 있습니다. 다중 모델 엔드포인트에 관한 자세한 내용은 [다중 모델 엔드포인트 호출](https://docs.aws.amazon.com/sagemaker/latest/dg/invoke-multi-model-endpoint.html)을 참조하십시오.

엔드포인트가 작동 중이면 Helper 함수는 엔드포인트 Amazon Resource Name(ARN)을 인쇄합니다. 엔드포인트 생성을 실행하려면 약 3~7분이 소요됩니다.

In [ ]:
# Create the endpoint
# Set the name of the endpoint. The name must be unique within an AWS Region in your AWS account.
endpoint_name = '{}-name'.format(endpoint_config_name)

create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name, 
    EndpointConfigName=endpoint_config_name
) 

def wait_for_endpoint_creation_complete(endpoint):
    """Helper function to wait for the completion of creating an endpoint"""
    response = sm.describe_endpoint(EndpointName=endpoint_name)
    status = response.get("EndpointStatus")
    while status == "Creating":
        print("Waiting for Endpoint Creation")
        time.sleep(15)
        response = sm.describe_endpoint(EndpointName=endpoint_name)
        status = response.get("EndpointStatus")

    if status != "InService":
        print(f"Failed to create endpoint, response: {response}")
        failureReason = response.get("FailureReason", "")
        raise SystemExit(
            f"Failed to create endpoint {create_endpoint_response['EndpointArn']}, status: {status}, reason: {failureReason}"
        )
    print(f"Endpoint {create_endpoint_response['EndpointArn']} successfully created.")

wait_for_endpoint_creation_complete(endpoint=create_endpoint_response)

## 과제 2.4: 엔드포인트 테스트 및 예측 생성

SageMaker 호스팅 서비스를 사용하여 모델을 배포한 후에는 엔드포인트로 테스트 데이터를 전송하여 해당 엔드포인트에서 모델을 테스트할 수 있습니다.

이 실습에서 사용하는 고객 레코드 중에는 소득이 50,000 USD 미만인 레코드(**income** 값 **1**)도 있고 50,000 USD 이상인 레코드(**income** 값 **0**)도 있습니다. 이러한 레코드가 포함된 엔드포인트를 호출하여 반환되는 점수를 확인합니다.

엔드포인트에서 실시간 예측을 확인하려는 경우 응답에서 반환되는 본문 텍스트를 읽습니다. 이 텍스트에는 예측 점수 목록이 포함되어 있습니다. 각 레코드의 점수는 **0**에서 **1** 사이입니다. 값이 **1**에 가까울수록 해당 고객의 소득이 50,000 USD 미만일 가능성이 높은 것입니다. 예를 들어 예측 점수가 **0.42**인 고객은 예측 점수가 **0.14**인 고객보다 소득이 50,000 USD 미만일 가능성이 높습니다. 정밀도, 재현율 및 F1 점수를 계산하려면 값은 0 또는 1로 반올림되고, 이전에 노트북에 저장한 대상 데이터 레이블과 비교됩니다.

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **참고:** **adult_data_processed_test_no_target.csv** 데이터 집합에 5,000개를 초과하는 레코드가 있습니다. 모든 실시간 추론 요청을 완료하는 데 2~3분이 소요됩니다.

In [ ]:
#get-model-A-predictions
predictions = ""

# set the cutoff value for the binary classification problem
cutoff_value = 0.5

# determine the classification based on a probability value 
def convert_probability_to_binary(probability):
    if probability >= cutoff_value:
        return "1"
    else:
        return "0"

print(f"Sending test traffic to the endpoint {endpoint_name}. \nPlease wait...")
with open("adult_data_processed_test_no_target.csv", "r") as f:
    for row in f:
        # print(".", end="", flush=True)
        payload = row.rstrip("\n")
        response = sm_runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="text/csv",
            Body=payload
        )

        pred_probability = response["Body"].read().decode("utf-8")
        new_prediction = convert_probability_to_binary(float(pred_probability))
        predictions = ",".join([predictions, new_prediction])
f.close()

# Convert the predictions to a numpy array and round the values (values closer to 0 round to 0, while values closer to 1 round to 1)
pred_np = np.fromstring(predictions[1:], sep=",")

print("Done!")

## 과제 2.5: 혼동 행렬로 예측 능력 평가

ML 모델은 실제 세계의 근사치에 해당하므로 일부 모델의 예측에 오류가 있을 가능성이 높습니다. 일부 응용 사례에서 모든 유형의 예측 오류는 동일한 영향을 미칩니다. 다른 응용 사례에서는 절대항이나 상대항에서, 달러로, 시간으로 또는 다른 방식으로 측정한 경우 한 가지 종류의 오류가 다른 오류보다 훨씬 더 비용이 많이 들거나 중대할 수 있습니다. ML에서 모델의 정확도는 올바른 예측 수를 총 예측 수로 나누어 정의합니다. 이 실습에서는 자격 있는 시민에게 정부 지원 서비스를 홍보할 수 있도록 소득이 50,000 USD 미만인 경우를 예측하려고 합니다.

혼동 행렬은 예측된 값과 실제 값을 비교합니다. 이 행렬은 관찰 결과의 예측 클래스와 실제 클래스를 비교하여 각 클래스에 대해 올바른 예측 및 잘못된 예측 수를 표로 나타냅니다. 코드 셀을 실행하면 **참 긍정(TP)**, **참 부정(TN)**, **거짓 긍정(FP)** 및 **거짓 부정(FN)** 값이 표시됩니다.

In [ ]:
#evaluate-prediction-power
# Configure the metrics for a confusion matrix with the original labels and the predictions
cf_matrix = confusion_matrix(df_labels, pred_np)
group_names = ['True Neg','False Pos','False Neg','True Pos']
group_counts = ["{0:0.0f}".format(value) for value in
                cf_matrix.flatten()]
group_percentages = ["{0:.2%}".format(value) for value in
                        cf_matrix.flatten()/np.sum(cf_matrix)]
labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
            zip(group_names,group_counts,group_percentages)]
labels = np.asarray(labels).reshape(2,2)

# Display the confusion matrix
sns.heatmap(cf_matrix, annot=labels, fmt='', cmap='Blues')

추론 결과를 평가할 때 모델이 얼마나 효과적이고 정확한지를 확인하기 위해 사용할 수 있는 몇 가지 측정값이 있습니다. 가장 일반적인 측정값은 **정확도**, **정밀도**, **재현율** 및 **F1 점수**입니다. 

- **정확도**: 참 긍정 수와 참 부정 수의 합계를 긍정 수와 부정 수를 더한 값으로 나누어 계산하는 측정값입니다. 계산 방식은 (TP + TN) / (P + N)입니다. 
- **정밀도**: 참 긍정 수를 참 긍정 수와 거짓 긍정 수를 더한 값으로 나누어 계산하는 측정값입니다. 계산 방식은 TP / (TP + FP)입니다.
- **재현율**: 참 긍정 수를 참 긍정 수와 거짓 부정 수를 더한 값으로 나누어 계산하는 측정값입니다. 계산 방식은 TP / (TP + FN)입니다.
- **F1 점수**: 정밀도와 재현율을 곱한 다음, 정밀도와 재현율을 더한 값으로 나누어 계산한 측정값입니다. 그런 다음, 결과에 2를 곱합니다. 계산 방식은 2 * ((정밀도 * 재현율)/(정밀도 + 재현율))입니다.

이 데이터 집합에 대해 높은 *정밀도*를 원할 것입니다. 변호 단체는 소득이 50,000 USD 미만인 모든 사람에게 지원을 제공할 수 있는 충분한 자금이 없습니다. 소득이 50,000 USD 미만인 개인을 높은 *정밀도*로 올바르게 식별하여 영향을 극대화하려고 합니다.

In [ ]:
# Calculate the F1 score
print('You can view a full classification report below:\n')

# Calculate the accuracy, precision, and recall of df_labels and predictions using sklearn
print(classification_report(df_labels, pred_np))

**1**로 시작하는 줄 옆에 표시된 **정밀도** 값은 무엇입니까? **재현율**이 **정밀도**보다 높거나 낮습니까? 

모델을 튜닝하려는 경우 목표는 재현율 결과가 감소함을 의미하더라도 정밀도를 계속 늘리는 것입니다. 정밀도와 재현율 간의 절충은 일반적인 ML 문제입니다. F1 점수는 F1 점수를 최대화하기 위해 작동하는 모델 및 사용 사례에 맞게 절충안에 가중치를 부여합니다. 이 사용 사례에서는 정밀도가 증가하는 한, F1 점수와 재현율이 모두 감소할 경우에 적절합니다.


## 과제 2.6: 엔드포인트 삭제

다음의 3단계를 수행하여 엔드포인트를 정리할 수 있습니다. 먼저 엔드포인트를 삭제합니다. 그런 다음, 엔드포인트 구성을 삭제합니다. 마지막으로, 배포한 모델이 더 이상 필요하지 않다면 모델을 삭제합니다.

In [ ]:
#delete-resources
# Delete endpoint
sm.delete_endpoint(EndpointName=endpoint_name)

# Delete endpoint configuration
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
                   
# Delete model
sm.delete_model(ModelName=model_name)


### 정리

이 노트북을 완료했습니다. 실습의 다음 부분으로 이동하려면 다음을 수행합니다.

- 이 노트북 파일을 닫습니다.
- 실습 세션으로 돌아가 **Conclusion** 부분을 계속 진행합니다.